1 - Abra uma pasta do projeto
2 - Inclua a tabela anexa a pasta
3 - Crie um notebook 
4 - Leia a tabela
5 - Crie uma tabela de população agregada por estado
6 - Organize essa tabela por população que mais cresceu entre 2022 e 2010 (Faça a diferença se necessário)
7 - Salve essa tabela nova em csv na pasta do seu projeto
8 - Agora trabalhe por município
9 - Organize essa tabela por população que mais cresceu entre 2022 e 2010 (Faça a diferença se necessário)
10 - Salve essa tabela nova em csv na pasta do seu projeto

In [1]:
import numpy as np
import pandas as pd


print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.5
numpy: 2.5.2


In [2]:
path = r"CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

In [3]:
bruto = pd.read_excel(path, sheet_name="Municípios", header=None)
bruto.head(10)

,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Censo Demográfico 2022: População e Domicílios...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
3,NaN,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
4,NaN,RO,11,00023,Ariquemes,90353,90353,96833
5,NaN,RO,11,00031,Cabixi,6313,6313,5351
6,NaN,RO,11,00049,Cacoal,78574,78574,86887
7,NaN,RO,11,00056,Cerejeiras,17029,17029,15890
8,NaN,RO,11,00064,Colorado do Oeste,18591,18591,15663
9,NaN,RO,11,00072,Corumbiara,8783,8783,7519


In [4]:
dados = bruto.iloc[3:, 1:].copy()
dados.columns = [
    "UF",
    "COD. UF",
    "COD. MUNIC",
    "NOME DO MUNICÍPIO",
    "População Município 2010 (Sinopse)",
    "População 2010 (Alterações de Limites até 2022)",
    "População Censo 2022",
]

dados = dados[dados["População Censo 2022"].notna()].copy()

for coluna in [
    "COD. UF",
    "COD. MUNIC",
    "População Município 2010 (Sinopse)",
    "População 2010 (Alterações de Limites até 2022)",
    "População Censo 2022",
]:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")

dados

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010 (Sinopse),População 2010 (Alterações de Limites até 2022),População Censo 2022
3,RO,11,15,Alta Floresta D'Oeste,24392,24392,21494
4,RO,11,23,Ariquemes,90353,90353,96833
5,RO,11,31,Cabixi,6313,6313,5351
6,RO,11,49,Cacoal,78574,78574,86887
7,RO,11,56,Cerejeiras,17029,17029,15890
...,...,...,...,...,...,...,...
5568,GO,52,22005,Vianópolis,12548,12548,14956
5569,GO,52,22054,Vicentinópolis,7371,7373,8768
5570,GO,52,22203,Vila Boa,4735,4735,4215
5571,GO,52,22302,Vila Propício,5145,5145,5815


## 1. População agregada por estado

In [5]:
dados["Diferença Populacional"] = (
    dados["População Censo 2022"]
    - dados["População 2010 (Alterações de Limites até 2022)"]
)

dados[["UF", "População 2010 (Alterações de Limites até 2022)", "População Censo 2022", "Diferença Populacional"]]

,UF,População 2010 (Alterações de Limites até 2022),População Censo 2022,Diferença Populacional
3,RO,24392,21494,-2898
4,RO,90353,96833,6480
5,RO,6313,5351,-962
6,RO,78574,86887,8313
7,RO,17029,15890,-1139
...,...,...,...,...
5568,GO,12548,14956,2408
5569,GO,7373,8768,1395
5570,GO,4735,4215,-520
5571,GO,5145,5815,670


In [6]:
populacao_estados = (
    dados.groupby("UF", as_index=False)
    .agg({
        "População 2010 (Alterações de Limites até 2022)": "sum",
        "População Censo 2022": "sum",
        "Diferença Populacional": "sum",
    })
    .sort_values("Diferença Populacional", ascending=False)
)

populacao_estados

,UF,População 2010 (Alterações de Limites até 2022),População Censo 2022,Diferença Populacional
25,SP,41262199,44411238,3149039
23,SC,6248436,7610361,1361925
8,GO,6001789,7056495,1054706
17,PR,10444526,11444380,999854
10,MG,19597330,20539989,942659
12,MT,3035122,3658649,623527
13,PA,7581051,8120131,539080
2,AM,3483985,3941613,457628
5,CE,8451644,8794957,343313
7,ES,3514952,3833712,318760


In [7]:
populacao_estados.to_csv("populacao_estados_2010_2022.csv", index=False, encoding="utf-8-sig")
print("Tabela por estado salva com sucesso!")

Tabela por estado salva com sucesso!


## 2. População por município

In [8]:
populacao_municipios = (
    dados[
        [
            "UF",
            "COD. UF",
            "COD. MUNIC",
            "NOME DO MUNICÍPIO",
            "População 2010 (Alterações de Limites até 2022)",
            "População Censo 2022",
            "Diferença Populacional",
        ]
    ]
    .sort_values("Diferença Populacional", ascending=False)
)

populacao_municipios

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População 2010 (Alterações de Limites até 2022),População Censo 2022,Diferença Populacional
114,AM,13,2603,Manaus,1802014,2063689,261675
5572,DF,53,108,Brasília,2572159,2817381,245222
3832,SP,35,50308,São Paulo,11253503,11451999,198496
3851,SP,35,52205,Sorocaba,586816,723682,136866
5420,GO,52,8707,Goiânia,1301912,1437366,135454
...,...,...,...,...,...,...,...
4934,RS,43,14902,Porto Alegre,1409351,1332845,-76506
172,PA,15,1402,Belém,1393399,1303403,-89996
3250,RJ,33,4904,São Gonçalo,999728,896744,-102984
3245,RJ,33,4557,Rio de Janeiro,6320446,6211223,-109223


In [9]:
populacao_municipios.to_csv("populacao_municipios_2010_2022.csv", index=False, encoding="utf-8-sig")
print("Tabela por município salva com sucesso!")

Tabela por município salva com sucesso!
